# Chapter 18 — Reinforcement Learning
Companion notebook to `part2_deep_learning/18_reinforcement_learning.md`.
Needs the `gymnasium` (or classic `gym`) package: `pip install gymnasium`.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from collections import deque

try:
    import gymnasium as gym
except ImportError:
    import gym

env = gym.make("CartPole-v1")
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

## Policy Gradients (REINFORCE)

In [ ]:
model = keras.models.Sequential([
    keras.layers.Dense(5, activation="elu", input_shape=[4]),
    keras.layers.Dense(1, activation="sigmoid"),
])

def play_one_step(env, obs, model, loss_fn):
    with tf.GradientTape() as tape:
        left_proba = model(obs[np.newaxis])
        action = (tf.random.uniform([1, 1]) > left_proba)
        y_target = tf.constant([[1.]]) - tf.cast(action, tf.float32)
        loss = tf.reduce_mean(loss_fn(y_target, left_proba))
    grads = tape.gradient(loss, model.trainable_variables)
    result = env.step(int(action[0, 0].numpy()))
    obs, reward, done = result[0], result[1], result[2]
    if len(result) > 3 and isinstance(result[3], bool):   # gymnasium: (obs, r, terminated, truncated, info)
        done = done or result[3]
    return obs, reward, done, grads

def play_multiple_episodes(env, n_episodes, n_max_steps, model, loss_fn):
    all_rewards, all_grads = [], []
    for episode in range(n_episodes):
        current_rewards, current_grads = [], []
        obs = env.reset()
        if isinstance(obs, tuple):   # gymnasium reset() returns (obs, info)
            obs = obs[0]
        for step in range(n_max_steps):
            obs, reward, done, grads = play_one_step(env, obs, model, loss_fn)
            current_rewards.append(reward)
            current_grads.append(grads)
            if done:
                break
        all_rewards.append(current_rewards)
        all_grads.append(current_grads)
    return all_rewards, all_grads

def discount_rewards(rewards, discount_factor):
    discounted = np.array(rewards, dtype=float)
    for step in range(len(rewards) - 2, -1, -1):
        discounted[step] += discounted[step + 1] * discount_factor
    return discounted

def discount_and_normalize_rewards(all_rewards, discount_factor):
    all_discounted_rewards = [discount_rewards(r, discount_factor) for r in all_rewards]
    flat = np.concatenate(all_discounted_rewards)
    mean, std = flat.mean(), flat.std()
    return [(dr - mean) / (std + 1e-8) for dr in all_discounted_rewards]

In [ ]:
n_iterations = 50
n_episodes_per_update = 10
n_max_steps = 200
discount_factor = 0.95
optimizer = keras.optimizers.Adam(learning_rate=0.01)
loss_fn = keras.losses.binary_crossentropy

for iteration in range(n_iterations):
    all_rewards, all_grads = play_multiple_episodes(env, n_episodes_per_update, n_max_steps, model, loss_fn)
    total_rewards = sum(map(sum, all_rewards))
    if iteration % 10 == 0:
        print(f"Iteration {iteration}: mean reward = {total_rewards / n_episodes_per_update:.1f}")
    all_final_rewards = discount_and_normalize_rewards(all_rewards, discount_factor)
    all_mean_grads = []
    for var_index in range(len(model.trainable_variables)):
        mean_grads = tf.reduce_mean(
            [final_reward * all_grads[ep][step][var_index]
             for ep, final_rewards in enumerate(all_final_rewards)
             for step, final_reward in enumerate(final_rewards)], axis=0)
        all_mean_grads.append(mean_grads)
    optimizer.apply_gradients(zip(all_mean_grads, model.trainable_variables))

## ✍️ Practice
1. Plot mean episode reward over training iterations — how many iterations until it's consistently near 200?
2. Explain the difference between on-policy (Policy Gradients) and off-policy (Q-Learning) learning.
3. Implement basic Deep Q-Learning with a replay buffer (see the notes for the full recipe) on CartPole. Do you see catastrophic forgetting?
4. Debug: DQN loss steadily decreases but game score gets worse — what should you track instead of loss?

In [ ]:
# Your code / notes here